# FanSphere AI — Stage 2: StatsBomb-only test

**Goal:** prove the on-pitch half of the pipeline works on real data — pull La Liga matches from StatsBomb's open dataset, compute engagement metrics, save CSVs.

**What this stage is NOT doing yet:** Reddit, sentiment scoring, database writes.

Total runtime: ~30–60 seconds (mostly waiting on HTTP calls to GitHub's raw JSON endpoint).

## 2.1 — What's in the StatsBomb open dataset?

`load_competitions()` returns every competition × season combination StatsBomb makes freely available. We filter to La Liga (`competition_id = 11`), which in their open data is **FC Barcelona's full La Liga match history across multiple seasons**.

In [ ]:
from src.load_statsbomb import load_competitions
import pandas as pd

competitions = load_competitions()
print(f'Total competition/season combinations available: {len(competitions)}')

la_liga = (
    competitions[competitions['competition_id'] == 11]
    .sort_values('season_name', ascending=False)
    [['competition_name', 'season_name', 'season_id']]
    .reset_index(drop=True)
)
print(f'La Liga seasons in StatsBomb open data: {len(la_liga)}\n')
la_liga.head(10)

## 2.2 — Load one season's matches

We grab the most recent La Liga season available. This is a single HTTP request to a JSON file on GitHub — ~38 matches.

In [ ]:
import requests
from src.load_statsbomb import STATSBOMB_BASE, _is_rivalry

season = la_liga.iloc[0]
print(f"Loading: {season['competition_name']} {season['season_name']}\n")

url = f"{STATSBOMB_BASE}/matches/11/{season['season_id']}.json"
raw = requests.get(url, timeout=30).json()

rows = []
for m in raw:
    home = m['home_team']['home_team_name']
    away = m['away_team']['away_team_name']
    rows.append({
        'match_id':    m['match_id'],
        'home_team':   home,
        'away_team':   away,
        'competition': season['competition_name'],
        'season':      season['season_name'],
        'match_date':  m['match_date'],
        'home_score':  m.get('home_score'),
        'away_score':  m.get('away_score'),
        'is_rivalry':  _is_rivalry(home, away),
    })

matches = pd.DataFrame(rows)
matches['match_date']  = pd.to_datetime(matches['match_date']).dt.date
matches['total_goals'] = matches['home_score'].fillna(0) + matches['away_score'].fillna(0)

print(f'Loaded {len(matches)} matches.')
matches.head(10)

## 2.3 — Rivalry detection

`load_statsbomb.RIVALRY_PAIRS` is a curated lookup (El Clásico, the North London derby, the Milan derby, etc.). Any fixture between a listed pair is flagged — these matches get a rivalry weight in the Match Hype Score later.

In [ ]:
rivalries = matches[matches['is_rivalry']][
    ['match_date', 'home_team', 'away_team', 'home_score', 'away_score']
]
print(f'Found {len(rivalries)} rivalry fixture(s) this season.\n')
rivalries

## 2.4 — Load goal events (10-match sample)

Goal events come from per-match JSON files — one HTTP request per match. To keep Stage 2 fast we sample **10 matches**: any rivalry games plus the highest-scoring ones. The production pipeline does this for every match.

Expect a progress bar and ~10–20 seconds wait.

In [ ]:
from src.load_statsbomb import load_goal_events, goals_per_match

sample = (
    matches
    .sort_values(['is_rivalry', 'total_goals'], ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print('Sample of 10 matches selected:')
print(sample[['home_team', 'away_team', 'is_rivalry', 'total_goals']].to_string())
print()

goal_events = load_goal_events(sample['match_id'])
print(f'\nLoaded {len(goal_events)} goal events.')
goal_events.head(10)

## 2.5 — Compute engagement metrics

We feed matches + goals + an empty sentiment DataFrame into `build_engagement_metrics`. Without Reddit data, comment/upvote inputs are zero — but the goal volume and rivalry weight still drive **excitement_index** and **match_hype_score**. This proves the maths runs on real data.

In [ ]:
from src.generate_metrics import build_engagement_metrics

# Empty sentiment DataFrame with the schema build_engagement_metrics expects
empty_sentiment = pd.DataFrame(columns=[
    'id', 'source', 'external_id', 'comment', 'upvotes', 'posted_at',
    'sentiment_score', 'sentiment_label', 'emotion', 'match_id'
])

goals = goals_per_match(goal_events)
engagement = build_engagement_metrics(sample, empty_sentiment, goals)

print(f'Computed metrics for {len(engagement)} matches.\n')
engagement

## 2.6 — Top matches by engagement score

The headline output: matches ranked by composite engagement. With no social data yet, rivalry matches and high-scoring games dominate — exactly the expected behaviour.

In [ ]:
result = sample.merge(engagement, on='match_id')

ranking = result[[
    'match_date', 'home_team', 'away_team', 'home_score', 'away_score',
    'is_rivalry', 'goal_events',
    'excitement_index', 'match_hype_score', 'engagement_score',
]].sort_values('engagement_score', ascending=False).reset_index(drop=True)

ranking

## 2.7 — Save outputs

Four CSVs land in `outputs/`. These are the same files the production pipeline emits — they'll eventually feed Power BI.

In [ ]:
from pathlib import Path

outputs = Path('outputs')
outputs.mkdir(exist_ok=True)

sample.to_csv(     outputs / 'stage2_matches.csv',      index=False)
goal_events.to_csv(outputs / 'stage2_goal_events.csv',  index=False)
engagement.to_csv( outputs / 'stage2_engagement.csv',   index=False)
ranking.to_csv(    outputs / 'stage2_ranking.csv',      index=False)

print('Saved to outputs/:')
for f in sorted(outputs.glob('stage2_*.csv')):
    print(f'  {f.name:30s} ({f.stat().st_size:>6,} bytes)')

## Stage 2 complete

Send me:

1. The output of cell **2.3** (rivalry fixtures detected this season)
2. The output of cell **2.6** (top-ranked matches table)
3. Confirmation that cell **2.7** saved 4 CSVs to `outputs/`

Then we move to **Stage 3** — Reddit credentials and live sentiment scoring on social chatter.